# UdaPlay Project — Part 2

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [14]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [15]:
import os
import json
import chromadb
from dotenv import load_dotenv
from pydantic import BaseModel
from tavily import TavilyClient
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [16]:
# Load environment variables
load_dotenv()
required = ["OPENAI_API_KEY", "OPENAI_BASE_URL", "TAVILY_API_KEY"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise RuntimeError(f"Missing environment variables: {', '.join(missing)}")
print("Environment variables loaded")

Environment variables loaded


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [17]:
@tool
def retrieve_game(query: str):
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry. 

    You'll receive results as list. Each element contains:
    - SourceId: id of the document in the internal database, use it to cite the source
    - Platform: like Game Boy, Playstation
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Publisher: Company that published the game
    - Description: Additional details about the game
    """
    chroma_client = chromadb.PersistentClient(path="chromadb")
    collection = chroma_client.get_collection("udaplay")
    results = collection.query(query_texts=[query], n_results=5)
    return [
        {
            "SourceId": doc_id,
            "Platform": game["Platform"],
            "Name": game["Name"],
            "YearOfRelease": game["YearOfRelease"],
            "Publisher": game["Publisher"],
            "Description": game["Description"],
        }
        for doc_id, game in zip(results["ids"][0], results["metadatas"][0])
    ]

#### Evaluate Retrieval Tool

In [18]:
class EvaluationResult(BaseModel):
    useful: bool
    description: str

@tool
def evaluate_retrieval(question: str, retrieved_docs: str):
    """
    Based on the user's question and on the list of retrieved documents, 
    it will analyze the usability of the documents to respond to that question. 
    args: 
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    llm = LLM(
        model="gpt-4o-mini",
        temperature=0,
        api_key=os.environ.get("OPENAI_API_KEY"),
    )
    llm_prompt = f"""
Your task is to evaluate if the documents are enough to respond to the query: "{question}".
The retrieved documents are: {retrieved_docs}

Set useful=true only if the documents contain the facts needed to answer the query.
Set useful=false if they are off-topic, incomplete or do not mention what the query asks about.
"""
    llm_response = llm.invoke([UserMessage(content=llm_prompt)], EvaluationResult)
    return json.loads(llm_response.content)

#### Game Web Search Tool

In [19]:
tavily_api_key = os.environ.get("TAVILY_API_KEY")

@tool
def game_web_search(question: str):
  """ search the web for information about games """
  client = TavilyClient(tavily_api_key)
  results = client.search(question, max_results=3)
  return [
      {"title": r["title"], "content": r["content"], "url": r["url"]}
      for r in results["results"]
  ]

### Agent

In [20]:
tools = [retrieve_game, evaluate_retrieval, game_web_search]

instructions = """You are a video game expert. Use the tools available to provide the most accurate answer about video game topics.
Follow this workflow for every question:
1. Call retrieve_game to search the internal video game database.
2. Call evaluate_retrieval with the original question and the documents you retrieved.
3. If evaluate_retrieval reports useful=false, call game_web_search to look the answer up on the web.
4. Answer concisely, then add a "Sources:" section citing where every fact came from:
   - for facts from the internal database, cite `internal database, document <SourceId> (<Name>)`
   - for facts from the web, cite the page title followed by its full URL
   Never state a fact without a matching source line.

You are in a multi-turn conversation. When the user refers back to something discussed earlier
("that game", "it", "the one you mentioned"), resolve the reference from the conversation history
before calling any tool, and pass the resolved, self-contained query to the tools.
"""

agent = Agent(
  model_name="gpt-4o",
  instructions=instructions,
  tools=tools,
  temperature=0,
)

### Running the agent

`report()` runs one turn and prints the whole trace: the agent's reasoning, every tool call with its
arguments, every tool result, and the final cited answer.

In [21]:
def _decode_tool_result(content: str) -> str:
    """Tool results are stored as json.dumps(str(result)) - unwrap the outer JSON string."""
    try:
        decoded = json.loads(content)
    except (json.JSONDecodeError, TypeError):
        return str(content)
    return decoded if isinstance(decoded, str) else json.dumps(decoded, indent=2)


def report(query: str, session_id: str = "default", max_result_chars: int = 700):
    """Run the agent and print its reasoning, every tool call/result, and the final answer."""
    run = agent.invoke(query, session_id=session_id)
    messages = run.get_final_state()["messages"]

    turn_start = max(i for i, m in enumerate(messages) if m.role == "user")
    turn = messages[turn_start + 1:]

    print("=" * 90)
    print(f"QUESTION: {query}")
    print(f"(session: {session_id})")
    print("=" * 90)

    step = 0
    for message in turn:
        if message.role == "assistant" and message.tool_calls:
            step += 1
            if message.content:
                print(f"\n--- Step {step}: reasoning ---\n{message.content}")
            for call in message.tool_calls:
                args = json.loads(call.function.arguments)
                print(f"\n--- Step {step}: tool call -> {call.function.name} ---")
                for key, value in args.items():
                    value = str(value)
                    if len(value) > max_result_chars:
                        value = value[:max_result_chars] + " ...[truncated]"
                    print(f"    {key}: {value}")

        elif message.role == "tool":
            result = _decode_tool_result(message.content)
            if len(result) > max_result_chars:
                result = result[:max_result_chars] + " ...[truncated]"
            print(f"\n--- tool result <- {message.name} ---\n{result}")

        elif message.role == "assistant":
            print(f"\n{'-' * 90}\nFINAL ANSWER:\n{message.content}\n{'-' * 90}\n")

    return run

#### Example query 1 - answered from the internal database

In [22]:
run_1 = report("When was Pokémon Gold and Silver released?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: When was Pokémon Gold and Silver released?
(session: default)

--- Step 1: tool call -> retrieve_game ---
    query: Pokémon Gold and Silver release date

--- tool result <- retrieve_game ---
[{'SourceId': '006', 'Platform': 'Game Boy Color', 'Name': 'Pokémon Gold and Silver', 'YearOfRelease': 1999, 'Publisher': 'Nintendo', 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.'}, {'SourceId': '007', 'Platform': 'Game Boy Advance', 'Name': 'Pokémon Ruby and Sapphire', 'YearOfRelease': 2002, 'Publisher': 'Nintendo', 'Description': 'Third-generation Pokémon games set in the Hoenn region

#### Example query 2 - answered from the internal database

In [23]:
run_2 = report("What was the first 3D platformer Mario game?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: What was the first 3D platformer Mario game?
(session: default)

--- Step 1: tool call -> retrieve_game ---
    query: first 3D platformer Mario game

--- tool result <- retrieve_game ---
[{'SourceId': '009', 'Platform': 'Nintendo 64', 'Name': 'Super Mario 64', 'YearOfRelease': 1996, 'Publisher': 'Nintendo', 'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach."}, {'SourceId': '008', 'Platform': 'Super Nintendo Entertainment System (SNES)', 'Name': 'Super Mario World', 'YearOfRelease': 1990, 'Publisher': 'Nintendo', 'Description': 'A classic platformer wh

#### Example query 3 - internal retrieval judged insufficient, falls back to web search

In [24]:
run_3 = report("Was Mortal Kombat X released for Playstation 5?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: Was Mortal Kombat X released for Playstation 5?
(session: default)

--- Step 1: tool call -> retrieve_game ---
    query: Mortal Kombat X release platforms

--- tool result <- retrieve_game ---
[{'SourceId': '015', 'Platform': 'Xbox Series X|S', 'Name': 'Halo Infinite', 'YearOfRelease': 2021, 'Publisher': 'Xbox Game Studios', 'Description': "The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting."}, {'SourceId': '012', 'Platform': 'Nintendo Switch', 'Name': 'Mario Kart 8 Deluxe', 'YearOfRelease':

### Conversation state across turns

All calls above ran in the same session (`default`), so the agent's short-term memory already holds
the full message history. The follow-up below contains no game name at all - it can only be answered
by resolving "that game" against the previous turns.

In [25]:
run_4 = report("Which publisher released that game, and what genre is it?")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: Which publisher released that game, and what genre is it?
(session: default)

------------------------------------------------------------------------------------------
FINAL ANSWER:
Mortal Kombat X was published by Warner Bros. Interactive Entertainment, and it is a fighting game.

Sources:
- [Mortal Kombat X - PlayStation Store](https://store.playstation.com/en-us/product/UP1018-CUSA00967_00-MORTALKOMBATX000)
- [Mortal Kombat X - Wikipedia](https://en.wikipedia.org/wiki/Mortal_Kombat_X)
------------------------------------------------------------------------------------------



In [26]:
# print the whole session info
runs = agent.get_session_runs("default")
print(f"Runs in session 'default': {len(runs)}")

history = runs[-1].get_final_state()["messages"]
print(f"Messages in the session: {len(history)}\n")
for message in history:
    if message.role == "user":
        print(f"  user      | {message.content}")
    elif message.role == "assistant" and message.tool_calls:
        names = ", ".join(c.function.name for c in message.tool_calls)
        print(f"  assistant | -> tool calls: {names}")
    elif message.role == "assistant":
        print(f"  assistant | {message.content.splitlines()[0][:80]}...")
    elif message.role == "tool":
        print(f"  tool      | <- {message.name}")
    elif message.role == "system":
        print(f"  system    | {message.content.splitlines()[0][:80]}...")

Runs in session 'default': 4
Messages in the session: 23

  system    | You are a video game expert. Use the tools available to provide the most accurat...
  user      | When was Pokémon Gold and Silver released?
  assistant | -> tool calls: retrieve_game
  tool      | <- retrieve_game
  assistant | -> tool calls: evaluate_retrieval
  tool      | <- evaluate_retrieval
  assistant | Pokémon Gold and Silver were released in 1999 for the Game Boy Color....
  user      | What was the first 3D platformer Mario game?
  assistant | -> tool calls: retrieve_game
  tool      | <- retrieve_game
  assistant | -> tool calls: evaluate_retrieval
  tool      | <- evaluate_retrieval
  assistant | The first 3D platformer Mario game was "Super Mario 64," released in 1996 for th...
  user      | Was Mortal Kombat X released for Playstation 5?
  assistant | -> tool calls: retrieve_game
  tool      | <- retrieve_game
  assistant | -> tool calls: evaluate_retrieval
  tool      | <- evaluate_retrieval
  assis